# LLM Few-Shot — Topic-Level Sentiment (Local Debug)

**Goal:** For each review, classify the sentiment *specifically about its assigned topic*.

**Key difference from document-level:**
- Document-level: overall positive/negative for the whole review
- Topic-level: sentiment about one specific aspect (e.g. shipping, sound quality)
- No ground truth → no F1 score, output is % positive/negative per topic

**This notebook:** ~5 samples per topic (70 total) on Mac via Ollama.

In [1]:
import re
import time
import requests
import pandas as pd
import numpy as np
from tqdm import tqdm

print('All imports OK')

All imports OK


In [2]:
# ── Configuration ──────────────────────────────────────────────────────────────

DEBUG_MODE          = True
SAMPLES_PER_TOPIC   = 5      # number of reviews to test per topic
RANDOM_STATE        = 42

# Ollama settings
OLLAMA_URL          = 'http://localhost:11434/api/generate'
OLLAMA_MODEL        = 'mistral'

DATA_PATH           = '../../guitars_with_topics_v2.parquet'
OUT_PREDICTIONS     = 'llm_topic_predictions_debug.csv'
OUT_TOPIC_SUMMARY   = 'llm_topic_sentiment_debug.csv'

print(f'Samples per topic: {SAMPLES_PER_TOPIC}')
print(f'Model: {OLLAMA_MODEL} via Ollama')

Samples per topic: 5
Model: mistral via Ollama


In [3]:
# ── Verify Ollama is running ───────────────────────────────────────────────────

def check_ollama():
    try:
        resp = requests.get('http://localhost:11434/api/tags', timeout=5)
        models = [m['name'] for m in resp.json().get('models', [])]
        print('Ollama running. Models:', models)
        if not any(OLLAMA_MODEL in m for m in models):
            print(f"WARNING: '{OLLAMA_MODEL}' not found. Run: ollama pull {OLLAMA_MODEL}")
        else:
            print(f"Model '{OLLAMA_MODEL}' is ready.")
    except Exception as e:
        print('ERROR: Cannot connect to Ollama. Run: ollama serve')
        raise e

check_ollama()

Ollama running. Models: ['mistral:latest']
Model 'mistral' is ready.


In [4]:
# ── Load data ──────────────────────────────────────────────────────────────────

df_raw = pd.read_parquet(DATA_PATH)
print(f'Raw: {len(df_raw):,} rows')

# Keep only rows with a valid topic and non-empty review text
df = df_raw.dropna(subset=['topic_label', 'text']).copy()
df['text'] = df['text'].astype(str).str.strip()
df = df[df['text'].str.len() > 10].reset_index(drop=True)

print(f'After filtering (topic + text not null): {len(df):,} rows')
print()
print('Topic distribution:')
print(df['topic_label'].value_counts().to_string())

Raw: 134,068 rows
After filtering (topic + text not null): 99,834 rows

Topic distribution:
topic_label
Beginner learning             13851
Fret / neck setup             13381
Accessories                    9420
Customer service / returns     9100
Pickups                        7852
Shipping damage                6543
String quality                 6198
Tuning stability               5907
Guitar size                    5753
Playability / chords           5576
Setup / action                 5224
Acoustic tone                  4511
Visual appearance              4149
Electronics / controls         2369


In [5]:
# ── Sample reviews for debug run ───────────────────────────────────────────────
# Take SAMPLES_PER_TOPIC reviews from each topic.
# We do NOT filter by rating here — topic-level has no ground truth labels.

sampled_parts = []
for topic, group in df.groupby('topic_label'):
    n = min(SAMPLES_PER_TOPIC, len(group))
    sampled_parts.append(group.sample(n=n, random_state=RANDOM_STATE))

eval_df = pd.concat(sampled_parts).reset_index(drop=True)
print(f'Debug eval set: {len(eval_df)} reviews across {eval_df["topic_label"].nunique()} topics')
print(eval_df['topic_label'].value_counts().to_string())

Debug eval set: 70 reviews across 14 topics
topic_label
Accessories                   5
Acoustic tone                 5
Beginner learning             5
Customer service / returns    5
Electronics / controls        5
Fret / neck setup             5
Guitar size                   5
Pickups                       5
Playability / chords          5
Setup / action                5
Shipping damage               5
String quality                5
Tuning stability              5
Visual appearance             5


In [6]:
# ── Few-shot examples for topic-level sentiment ────────────────────────────────
# Key design: use the SAME review with TWO different topics to show the model
# that it must focus only on the given topic, not the overall review sentiment.

FEW_SHOT_EXAMPLES = [
    # Example 1 & 2: same review, different topics → different sentiment
    {
        'topic':  'Shipping damage',
        'review': 'The guitar sounds absolutely incredible, but it arrived with a cracked headstock '
                  'and the box was completely crushed. Clearly it was not packaged properly.',
        'label':  'negative'
    },
    {
        'topic':  'Acoustic tone',
        'review': 'The guitar sounds absolutely incredible, but it arrived with a cracked headstock '
                  'and the box was completely crushed. Clearly it was not packaged properly.',
        'label':  'positive'
    },
    # Example 3: clearly positive about tuning
    {
        'topic':  'Tuning stability',
        'review': 'I have been playing this guitar every day for three months and it holds its tuning '
                  'remarkably well. Even after aggressive strumming it stays in tune.',
        'label':  'positive'
    },
    # Example 4: clearly negative about setup
    {
        'topic':  'Setup / action',
        'review': 'The action is extremely high right out of the box. It is very hard to press the strings '
                  'down, especially on the higher frets. Needs a full professional setup before it is playable.',
        'label':  'negative'
    },
    # Example 5: review is mostly negative overall, but positive about the specific topic
    {
        'topic':  'Visual appearance',
        'review': 'Very disappointed with this purchase. The frets are sharp and the action is too high. '
                  'However, I have to admit it looks stunning — the sunburst finish is beautiful.',
        'label':  'positive'
    },
]

print(f'{len(FEW_SHOT_EXAMPLES)} few-shot examples ready.')
for ex in FEW_SHOT_EXAMPLES:
    print(f"  [{ex['label']:8s}] topic={ex['topic']} | {ex['review'][:80]}...")

5 few-shot examples ready.
  [negative] topic=Shipping damage | The guitar sounds absolutely incredible, but it arrived with a cracked headstock...
  [positive] topic=Acoustic tone | The guitar sounds absolutely incredible, but it arrived with a cracked headstock...
  [positive] topic=Tuning stability | I have been playing this guitar every day for three months and it holds its tuni...
  [negative] topic=Setup / action | The action is extremely high right out of the box. It is very hard to press the ...
  [positive] topic=Visual appearance | Very disappointed with this purchase. The frets are sharp and the action is too ...


In [7]:
# ── Prompt builder ─────────────────────────────────────────────────────────────

SYSTEM_PROMPT = (
    'You are a sentiment classifier specializing in guitar product reviews. '
    'You will be given a review and a specific topic. '
    'Your task is to determine the sentiment expressed in the review SPECIFICALLY ABOUT that topic. '
    'Ignore the overall tone of the review — focus only on what the reviewer says about the given topic. '
    'You MUST respond with exactly one word: positive or negative. '
    'No explanation. No punctuation. Just one word.'
)

def build_topic_prompt(review_text: str, topic: str) -> str:
    """Build a few-shot prompt for topic-level sentiment classification."""
    lines = [SYSTEM_PROMPT, '']

    for ex in FEW_SHOT_EXAMPLES:
        ex_text = ex['review'][:300] + ('...' if len(ex['review']) > 300 else '')
        lines.append(f'Topic: {ex["topic"]}')
        lines.append(f'Review: {ex_text}')
        lines.append(f'Sentiment about "{ex["topic"]}": {ex["label"]}')
        lines.append('')

    truncated = review_text[:400] + ('...' if len(review_text) > 400 else '')
    lines.append(f'Topic: {topic}')
    lines.append(f'Review: {truncated}')
    lines.append(f'Sentiment about "{topic}":')

    return '\n'.join(lines)

# Sanity check
sample = build_topic_prompt(
    'The packaging was great but the guitar buzzes on every fret.',
    'Fret / neck setup'
)
print(sample)
print(f'\nPrompt length: {len(sample)} chars')

You are a sentiment classifier specializing in guitar product reviews. You will be given a review and a specific topic. Your task is to determine the sentiment expressed in the review SPECIFICALLY ABOUT that topic. Ignore the overall tone of the review — focus only on what the reviewer says about the given topic. You MUST respond with exactly one word: positive or negative. No explanation. No punctuation. Just one word.

Topic: Shipping damage
Review: The guitar sounds absolutely incredible, but it arrived with a cracked headstock and the box was completely crushed. Clearly it was not packaged properly.
Sentiment about "Shipping damage": negative

Topic: Acoustic tone
Review: The guitar sounds absolutely incredible, but it arrived with a cracked headstock and the box was completely crushed. Clearly it was not packaged properly.
Sentiment about "Acoustic tone": positive

Topic: Tuning stability
Review: I have been playing this guitar every day for three months and it holds its tuning re

In [8]:
# ── Ollama inference ───────────────────────────────────────────────────────────

def parse_label(raw: str) -> str:
    text = raw.strip().lower()
    first = re.split(r'[^a-z]', text)[0]
    if first in ('positive', 'pos'): return 'positive'
    if first in ('negative', 'neg'): return 'negative'
    if 'positive' in text: return 'positive'
    if 'negative' in text: return 'negative'
    return 'N/A'

def query_ollama(prompt: str) -> tuple[str, str]:
    payload = {
        'model':  OLLAMA_MODEL,
        'prompt': prompt,
        'stream': False,
        'options': {'temperature': 0.0, 'num_predict': 10, 'top_p': 1.0}
    }
    try:
        resp = requests.post(OLLAMA_URL, json=payload, timeout=120)
        resp.raise_for_status()
        raw = resp.json().get('response', '').strip()
        return raw, parse_label(raw)
    except requests.exceptions.Timeout:
        return 'TIMEOUT', 'N/A'
    except Exception as e:
        return f'ERROR:{e}', 'N/A'

# Quick test
test_prompt = build_topic_prompt(
    'Strings snapped after two days. Terrible quality.',
    'String quality'
)
raw, label = query_ollama(test_prompt)
print(f'Raw: "{raw}" → Parsed: {label}')
print('Expected: negative')

Raw: "negative" → Parsed: negative
Expected: negative


In [9]:
# ── Run inference ──────────────────────────────────────────────────────────────

print(f'Running topic-level inference on {len(eval_df)} reviews...')
start = time.time()

raw_outputs  = []
pred_labels  = []

for _, row in tqdm(eval_df.iterrows(), total=len(eval_df), desc='Classifying'):
    prompt = build_topic_prompt(row['text'], row['topic_label'])
    raw, label = query_ollama(prompt)
    raw_outputs.append(raw)
    pred_labels.append(label)

elapsed = time.time() - start
print(f'Done in {elapsed:.1f}s ({elapsed/len(eval_df):.1f}s per review)')

eval_df = eval_df.copy()
eval_df['raw_output']     = raw_outputs
eval_df['pred_sentiment'] = pred_labels

print('\nPrediction distribution:')
print(eval_df['pred_sentiment'].value_counts())

na_count = (eval_df['pred_sentiment'] == 'N/A').sum()
print(f'N/A rate: {na_count}/{len(eval_df)} ({100*na_count/len(eval_df):.1f}%)')

Running topic-level inference on 70 reviews...


Classifying: 100%|██████████| 70/70 [02:06<00:00,  1.81s/it]


Done in 127.0s (1.8s per review)

Prediction distribution:
pred_sentiment
positive    43
negative    18
N/A          9
Name: count, dtype: int64
N/A rate: 9/70 (12.9%)


In [10]:
# ── Inspect results per topic ──────────────────────────────────────────────────
# No ground truth → we check manually whether results look sensible

valid_df = eval_df[eval_df['pred_sentiment'] != 'N/A'].copy()

print('=== Per-topic predictions (debug sample) ===')
print()
for topic, group in valid_df.groupby('topic_label'):
    counts   = group['pred_sentiment'].value_counts()
    pos      = counts.get('positive', 0)
    neg      = counts.get('negative', 0)
    total    = pos + neg
    neg_pct  = 100 * neg / total if total > 0 else 0
    print(f'{topic:<30} positive={pos}  negative={neg}  neg%={neg_pct:.0f}%')

=== Per-topic predictions (debug sample) ===

Accessories                    positive=2  negative=1  neg%=33%
Acoustic tone                  positive=5  negative=0  neg%=0%
Beginner learning              positive=5  negative=0  neg%=0%
Customer service / returns     positive=0  negative=2  neg%=100%
Electronics / controls         positive=4  negative=1  neg%=20%
Fret / neck setup              positive=3  negative=2  neg%=40%
Guitar size                    positive=3  negative=1  neg%=25%
Pickups                        positive=4  negative=0  neg%=0%
Playability / chords           positive=4  negative=1  neg%=20%
Setup / action                 positive=5  negative=0  neg%=0%
Shipping damage                positive=0  negative=5  neg%=100%
String quality                 positive=0  negative=3  neg%=100%
Tuning stability               positive=3  negative=2  neg%=40%
Visual appearance              positive=5  negative=0  neg%=0%


In [11]:
# ── Manual quality check: read a few predictions ───────────────────────────────
# Pick a topic and look at individual reviews + predictions to verify they make sense

CHECK_TOPIC = 'Shipping damage'   # change this to any topic you want to inspect

subset = eval_df[eval_df['topic_label'] == CHECK_TOPIC]
print(f'=== Manual check: {CHECK_TOPIC} ===')
for _, row in subset.iterrows():
    print(f"Pred: {row['pred_sentiment']}  |  Raw: '{row['raw_output']}'")
    print(f"Review: {row['text'][:200]}")
    print()

=== Manual check: Shipping damage ===
Pred: negative  |  Raw: 'negative'
Review: The guitar is a great guitar. I already have one blue and that's why I bought one in red. I really love the ruby red color. I was psyched to get it! I waited outside for the delivery guy. And when you

Pred: negative  |  Raw: 'negative'
Review: Had it for 3 days. Opened it day it was delivered. The strings were very loose so I tightened them and tuned it as close as possible being it is a children's guitar. Today the part that holds the stri

Pred: negative  |  Raw: 'negative'
Review: I was able to open it carefully and was happy to see everything in good shape. However the only cons about the guitar is that it had a small chipped on the side. Overall it still works and looks compl

Pred: negative  |  Raw: 'Negative (implied)'
Review: Guitar quality was as expected, but packaging was not. Last time I bought a Yamaha guitar from another on-line music store, they added another layer of packaging.

Pred: nega

In [12]:
# ── Topic-level sentiment summary ──────────────────────────────────────────────

summary_rows = []
for topic, group in valid_df.groupby('topic_label'):
    counts   = group['pred_sentiment'].value_counts()
    pos      = counts.get('positive', 0)
    neg      = counts.get('negative', 0)
    total    = pos + neg
    summary_rows.append({
        'topic':        topic,
        'total_reviews': total,
        'pct_positive': round(100 * pos / total, 1) if total > 0 else None,
        'pct_negative': round(100 * neg / total, 1) if total > 0 else None,
    })

topic_summary = pd.DataFrame(summary_rows).sort_values('pct_negative', ascending=False)
print('Topic-level sentiment summary (debug):')
print(topic_summary.to_string(index=False))

Topic-level sentiment summary (debug):
                     topic  total_reviews  pct_positive  pct_negative
Customer service / returns              2           0.0         100.0
           Shipping damage              5           0.0         100.0
            String quality              3           0.0         100.0
         Fret / neck setup              5          60.0          40.0
          Tuning stability              5          60.0          40.0
               Accessories              3          66.7          33.3
               Guitar size              4          75.0          25.0
    Electronics / controls              5          80.0          20.0
      Playability / chords              5          80.0          20.0
             Acoustic tone              5         100.0           0.0
         Beginner learning              5         100.0           0.0
                   Pickups              4         100.0           0.0
            Setup / action              5         1

In [13]:
# ── Save results ───────────────────────────────────────────────────────────────

save_cols = ['topic_label', 'text', 'pred_sentiment', 'raw_output']
eval_df[save_cols].to_csv(OUT_PREDICTIONS, index=False)
print(f'Per-review predictions → {OUT_PREDICTIONS}')

topic_summary.to_csv(OUT_TOPIC_SUMMARY, index=False)
print(f'Topic summary → {OUT_TOPIC_SUMMARY}')
print()
print('Debug run complete! If results look sensible, proceed to HPC full run.')

Per-review predictions → llm_topic_predictions_debug.csv
Topic summary → llm_topic_sentiment_debug.csv

Debug run complete! If results look sensible, proceed to HPC full run.
